# RefugeeReach — Gemma 4 Live Demo
### Kaggle Gemma 4 Hackathon · Main Track · Digital Equity & Inclusivity · Ollama Special Technology

---

**110 million people are forcibly displaced right now.**  
Every one must register. Most cannot read the form. There is no interpreter. And in a camp, there is no internet.

**RefugeeReach** is a multilingual, offline-first humanitarian intake assistant powered by **Gemma 4 E4B**.  
It runs entirely on local hardware — no API key, no cloud dependency.

---

## What this notebook demonstrates

| Feature | What Gemma 4 does |
|---------|-------------------|
| **Multilingual Intake Chat** | Converses in Arabic, Ukrainian, Dari, French, English — detects language automatically |
| **Document Vision** | Reads passport / ID / medical doc images → extracts structured fields |
| **Document Translation** | Explains any foreign document in the person's language |
| **Medical Handoff** | Generates structured triage note with urgency level for clinical staff |
| **Skills Matching** | Matches displaced person's background to humanitarian opportunities |
| **PDF Export** | Generates a dignified bilingual case summary for caseworkers |

---

## Before running

1. **Enable GPU**: Notebook Settings → Accelerator → **T4 GPU** (required — model is 4B params)
2. **Enable Internet**: Notebook Settings → Internet → On
3. **Model access**: Either add Gemma 4 via `+ Add data → Models → Gemma 4` in Kaggle,  
   OR add your HuggingFace token to Kaggle Secrets as `HUGGINGFACE_TOKEN` and accept Gemma terms at [hf.co/google/gemma-4-e4b-it](https://huggingface.co/google/gemma-4-e4b-it)
4. **Run All** — the Gradio URL appears in the last cell (~5 min total)

---

## Demo story: Amira

Try this walkthrough when the UI opens:
1. **Welcome tab** → Select Arabic → Start New Case
2. **Intake Chat** → type: `مرحباً، اسمي أميرة حسن` (Hello, my name is Amira Hassan)
3. **Document Vision** → upload any passport image → Extract Fields
4. **Medical** → type symptoms: `persistent cough, night sweats, prior TB exposure`
5. **Skills** → roles: `pediatric nurse, medical triage` → Find Matches
6. **Export** → Generate PDF → download the case summary

In [ ]:
# ── 1. Install dependencies ───────────────────────────────────────────────────
!pip install -q \
    "gradio>=4.44" \
    "transformers>=4.47" \
    "accelerate>=0.30" \
    bitsandbytes \
    fpdf2 \
    Pillow \
    arabic-reshaper \
    python-bidi \
    huggingface_hub
print("✓ Dependencies installed")

In [ ]:
# ── 2. Imports & device check ─────────────────────────────────────────────────
import os, json, uuid, io, re, urllib.request
from datetime import datetime
from pathlib import Path
import torch
from PIL import Image
import gradio as gr
from fpdf import FPDF

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__} | Device: {DEVICE}")
if DEVICE == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} | VRAM: {props.total_memory / 1e9:.1f} GB")
else:
    print("⚠ No GPU detected — inference will be slow. Enable T4 GPU in Notebook Settings.")

In [ ]:
# ── 3. Load Gemma 4 E4B ───────────────────────────────────────────────────────
#
# Tries Kaggle-hosted model first (no auth), then HuggingFace Hub.
# If using HF Hub, add HUGGINGFACE_TOKEN to Kaggle Secrets and accept
# Gemma terms at https://huggingface.co/google/gemma-4-e4b-it

from transformers import AutoProcessor, AutoModelForCausalLM

_KAGGLE_PATHS = [
    "/kaggle/input/gemma/transformers/gemma-4-e4b-it/1",
    "/kaggle/input/gemma4/transformers/gemma-4-e4b-it/1",
    "/kaggle/input/google-gemma-4/transformers/gemma-4-e4b-it/1",
]
_HF_ID = "google/gemma-4-e4b-it"

def _resolve_model_path():
    for p in _KAGGLE_PATHS:
        if os.path.exists(p):
            print(f"✓ Found Kaggle-hosted model: {p}")
            return p
    print(f"Kaggle model not found — loading from HuggingFace Hub: {_HF_ID}")
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("HUGGINGFACE_TOKEN")
        from huggingface_hub import login
        login(token=token, add_to_git_credential=False)
        print("✓ HuggingFace login OK")
    except Exception as e:
        print(f"⚠ No HUGGINGFACE_TOKEN in Kaggle Secrets ({e})")
        print("  Model may fail to load if it requires authentication.")
    return _HF_ID

MODEL_PATH = _resolve_model_path()

print(f"\nLoading processor...")
processor = AutoProcessor.from_pretrained(MODEL_PATH)

print("Loading model in 4-bit (fits T4 16 GB VRAM)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    load_in_4bit=True,
)
model.eval()
print("\n✓ Gemma 4 E4B ready")

In [ ]:
# ── 4. Inference helpers ──────────────────────────────────────────────────────

def _strip_fences(s: str) -> str:
    """Remove markdown code fences Gemma 4 sometimes wraps JSON in."""
    s = s.strip()
    if s.startswith("```"):
        lines = s.splitlines()
        inner = lines[1:]
        if inner and inner[-1].strip() == "```":
            inner = inner[:-1]
        s = "\n".join(inner).strip()
    return s


def generate_text(messages: list, max_new_tokens: int = 512) -> str:
    """Run text-only inference with conversation history."""
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = processor(text=text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )
    generated = out[0][inputs["input_ids"].shape[1]:]
    return processor.decode(generated, skip_special_tokens=True).strip()


def generate_with_image(pil_image: Image.Image, prompt: str, max_new_tokens: int = 512) -> str:
    """Run vision+text inference with a PIL image."""
    messages = [{
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": prompt},
        ]
    }]
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = processor(
        text=text,
        images=[pil_image],
        return_tensors="pt",
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )
    generated = out[0][inputs["input_ids"].shape[1]:]
    return processor.decode(generated, skip_special_tokens=True).strip()


print("✓ Inference helpers ready")

In [ ]:
# ── 5. Download Unicode font for PDF Arabic rendering ─────────────────────────
#
# fpdf2's built-in Helvetica is Latin-1 only.
# We download Noto Sans (Latin) + Noto Naskh Arabic for the PDF client copy.

FONT_DIR = Path("/tmp/refugeereach_fonts")
FONT_DIR.mkdir(exist_ok=True)

_FONTS = {
    "NotoSans": (
        "https://github.com/google/fonts/raw/main/ofl/notosans/NotoSans-Regular.ttf",
        FONT_DIR / "NotoSans-Regular.ttf",
    ),
    "NotoArabic": (
        "https://github.com/google/fonts/raw/main/ofl/notosansarabic/NotoSansArabic-Regular.ttf",
        FONT_DIR / "NotoSansArabic-Regular.ttf",
    ),
}

for name, (url, dest) in _FONTS.items():
    if not dest.exists():
        print(f"Downloading {name}...")
        try:
            urllib.request.urlretrieve(url, dest)
            print(f"  ✓ {dest.name}")
        except Exception as e:
            print(f"  ⚠ Could not download {name} ({e}) — PDF will use Latin fallback")
    else:
        print(f"✓ {name} already cached")

NOTO_SANS = str(_FONTS["NotoSans"][1]) if _FONTS["NotoSans"][1].exists() else None
NOTO_ARABIC = str(_FONTS["NotoArabic"][1]) if _FONTS["NotoArabic"][1].exists() else None
print(f"\nUnicode fonts: NotoSans={'yes' if NOTO_SANS else 'no'}, NotoArabic={'yes' if NOTO_ARABIC else 'no'}")

In [ ]:
# ── 6. Prompts (mirrored from models/prompts/*.md) ────────────────────────────

INTAKE_SYSTEM = """You are a compassionate, multilingual intake assistant for displaced people at a humanitarian reception center.

Your role is to help gather the information needed for registration, medical support, and skills matching — in a calm, dignified, and accessible way.

Behavior rules:
- Always respond in the user's language. Detect it automatically.
- Ask one question at a time. Never overwhelm the user.
- Use simple, plain language. Avoid jargon and bureaucratic phrasing.
- If the user seems distressed, acknowledge their situation with empathy before continuing.
- Never invent or assume information that was not provided.
- Never provide final legal or medical advice.

Required fields to collect:
- Full name, date of birth, nationality, gender
- Family size (number of people traveling with them)
- Current location or camp assignment
- Any vulnerability indicators (unaccompanied minor, pregnancy, disability, urgent medical need)"""

DOCUMENT_EXTRACTION_PROMPT = """You are a humanitarian document assistant. Extract structured information from this document image.

Rules:
1. Only extract field values clearly visible in the image. Never invent or guess.
2. If a field is not visible, set its value to null and confidence to 0.0.
3. Return ONLY a single valid JSON object — no other text, no markdown.

Return this schema:
{"document_type": "passport|national_id|medical|legal_notice|other",
 "fields": {
   "full_name":       {"value": null, "confidence": 0.0, "source_text": null},
   "date_of_birth":   {"value": null, "confidence": 0.0, "source_text": null},
   "nationality":     {"value": null, "confidence": 0.0, "source_text": null},
   "document_number": {"value": null, "confidence": 0.0, "source_text": null},
   "issue_date":      {"value": null, "confidence": 0.0, "source_text": null},
   "expiry_date":     {"value": null, "confidence": 0.0, "source_text": null},
   "place_of_birth":  {"value": null, "confidence": 0.0, "source_text": null},
   "gender":          {"value": null, "confidence": 0.0, "source_text": null}
 },
 "summary": "1-2 sentence plain-language explanation of what this document is.",
 "overall_confidence": 0.0}"""

MEDICAL_HANDOFF_TEMPLATE = """You are a clinical intake support assistant preparing a handoff note for a healthcare worker.
You are NOT making a diagnosis. You are summarizing so the clinician can triage efficiently.

Symptoms: {symptoms}
Existing conditions: {existing_conditions}
Medications mentioned: {medications}

Instructions:
1. Summarize in plain clinical language (2-3 sentences).
2. List any medications or conditions mentioned — do not add any not provided.
3. Assign urgency: "urgent" (immediate attention), "elevated" (same-day), or "routine" (stable).
4. Note any missing information the clinician should verify.

Respond with valid JSON only:
{"summary_for_staff": "...", "urgency_level": "routine|elevated|urgent", "missing_info": [...]}"""

MEDICAL_VISION_PROMPT = """Analyze this medical document image at a humanitarian reception center.
Extract: conditions, medications, vaccination history, treatment dates, and urgent flags.
Write a 2-3 sentence summary for a clinician. Assign urgency: urgent, elevated, or routine.
Do not invent information not visible in the document.

Respond with valid JSON only:
{"document_type": "...", "conditions": [], "medications": [], "vaccinations": [],
 "summary_for_staff": "...", "urgency_level": "routine", "missing_info": []}"""

_LANG_NAMES = {
    "ar": "Arabic", "uk": "Ukrainian", "fa": "Dari (Farsi)",
    "fr": "French", "en": "English",
}

OPPORTUNITY_CATALOG = [
    {"title": "Registered Nurse — Camp Medical Unit",
     "description": "Qualified nurses and midwives needed urgently for the camp medical unit. Responsibilities include patient triage, wound care, medication administration, and supporting the camp doctor. Nursing degree or equivalent clinical experience required. Arabic, French, or English speakers prioritised.",
     "location": "Camp Medical Center, Zone A"},
    {"title": "Pediatric Healthcare Assistant",
     "description": "Support pediatric nurses and doctors with child health assessments, vaccination record review, and growth monitoring. Prior nursing, midwifery, or pediatric care experience essential. Positions available immediately.",
     "location": "UNICEF Children's Health Clinic, Zone B"},
    {"title": "Community Health Worker Assistant",
     "description": "Support trained health workers with patient registration, translation, and basic triage at the camp medical center. Prior nursing, paramedic, or medical experience preferred.",
     "location": "Camp Medical Center, Zone A"},
    {"title": "Interpreter / Language Support Volunteer",
     "description": "Assist NGO staff as an interpreter for Arabic, Dari, Ukrainian, or French speakers during registration and legal orientation sessions. Bilingual applicants with any professional background welcome.",
     "location": "Reception Center, Main Hall"},
    {"title": "Primary School Teaching Assistant",
     "description": "Support UNHCR education partner classrooms for children aged 6-12. Qualified teachers and educators prioritized. Language support a plus.",
     "location": "Camp School, Zone C"},
    {"title": "Legal Aid Intake Support",
     "description": "Assist legal aid volunteers by helping clients complete intake forms and understand their rights under asylum procedures. No legal background required.",
     "location": "Legal Aid Clinic, Zone A"},
    {"title": "IT and Digital Literacy Trainer",
     "description": "Teach basic computer and smartphone skills to community members. Experience in IT, computing, or education preferred.",
     "location": "Community Center, Zone D"},
    {"title": "Food Distribution Support",
     "description": "Assist WFP food distribution teams with sorting, packing, and record-keeping. Physical fitness required.",
     "location": "Distribution Point 3, Zone B"},
    {"title": "Vocational Training — Construction Skills",
     "description": "4-week certified training in basic construction, shelter maintenance, and WASH facility repair. Open to adults 18+.",
     "location": "ILO Training Center"},
    {"title": "Child Nutrition Program Assistant",
     "description": "Help weigh and measure children, record intake data, and support community health workers delivering nutritional supplements. Healthcare background an advantage.",
     "location": "UNICEF Nutrition Tent, Zone B"},
]

print("✓ Prompts and catalog loaded")

In [ ]:
# ── 7. Feature functions ──────────────────────────────────────────────────────

def new_case(language: str) -> dict:
    """Create a fresh in-memory case record (replaces DB in this notebook)."""
    return {
        "case_id": str(uuid.uuid4())[:8].upper(),
        "language": language,
        "person_name": None,
        "date_of_birth": None,
        "nationality": None,
        "gender": None,
        "family_size": None,
        "current_location": None,
        "vulnerability_flags": [],
        "notes": None,
        "medical": None,
        "skills_matches": [],
    }


def intake_turn(user_msg: str, messages: list, case: dict) -> tuple:
    """One intake conversation turn. Returns (reply, updated_messages, updated_case)."""
    if not messages:
        messages = [{"role": "system", "content": INTAKE_SYSTEM}]

    messages = messages + [{"role": "user", "content": user_msg}]
    reply = generate_text(messages, max_new_tokens=350)
    messages = messages + [{"role": "assistant", "content": reply}]

    # After a few turns, extract structured fields from conversation
    if len(messages) >= 7:  # system + 3 user/assistant pairs
        case = _extract_case_fields(messages, case)

    return reply, messages, case


def _extract_case_fields(messages: list, case: dict) -> dict:
    """Ask Gemma 4 to extract structured fields from conversation history."""
    conversation_text = "\n".join(
        f"{m['role'].upper()}: {m['content']}"
        for m in messages
        if m["role"] in ("user", "assistant")
    )
    extract_prompt = [
        {"role": "user", "content": (
            f"Extract registration fields from this intake conversation.\n"
            f"Return ONLY valid JSON, no other text.\n"
            f"Use null for any field not clearly mentioned.\n\n"
            f"Conversation:\n{conversation_text}\n\n"
            f'Return: {{"person_name": null, "date_of_birth": null, "nationality": null, '
            f'"gender": null, "family_size": null, "current_location": null, "notes": null}}'
        )}
    ]
    try:
        raw = generate_text(extract_prompt, max_new_tokens=200)
        fields = json.loads(_strip_fences(raw))
        for k, v in fields.items():
            if v is not None and k in case:
                case[k] = v
    except Exception:
        pass
    return case


def extract_document(pil_image: Image.Image) -> dict:
    """Vision: extract structured fields from a document image."""
    if pil_image is None:
        return {"error": "No image provided"}
    raw = generate_with_image(pil_image, DOCUMENT_EXTRACTION_PROMPT, max_new_tokens=600)
    try:
        parsed = json.loads(_strip_fences(raw))
        if "fields" not in parsed:
            parsed = {"document_type": "other", "fields": {}, "summary": raw, "overall_confidence": 0.0}
    except json.JSONDecodeError:
        parsed = {"document_type": "other", "fields": {}, "summary": raw, "overall_confidence": 0.0}
    return parsed


def translate_document(pil_image: Image.Image, target_lang: str) -> str:
    """Vision: explain any document in the person's language."""
    if pil_image is None:
        return "No image provided."
    lang_name = _LANG_NAMES.get(target_lang, "English")
    prompt = (
        f"You are a compassionate humanitarian interpreter. "
        f"A displaced person has handed you this document and cannot read it. "
        f"Read it carefully, then write a clear explanation entirely in {lang_name}.\n\n"
        f"Your response must cover:\n"
        f"1. What type of document this is (one sentence)\n"
        f"2. What the document says in plain, simple language\n"
        f"3. What the person needs to do or know as a result\n"
        f"4. One warm, reassuring closing sentence\n\n"
        f"Write ONLY in {lang_name}. Be kind and simple."
    )
    return generate_with_image(pil_image, prompt, max_new_tokens=500)


def medical_handoff_text(symptoms: str, conditions: str, medications: str) -> dict:
    """Generate structured medical handoff from text description."""
    if not symptoms.strip():
        return {"error": "Please describe the symptoms."}
    prompt = (
        MEDICAL_HANDOFF_TEMPLATE
        .replace("{symptoms}", symptoms)
        .replace("{existing_conditions}", conditions or "None reported")
        .replace("{medications}", medications or "None reported")
    )
    raw = generate_text([{"role": "user", "content": prompt}], max_new_tokens=400)
    try:
        parsed = json.loads(_strip_fences(raw))
        if isinstance(parsed.get("urgency_level"), str):
            parsed["urgency_level"] = parsed["urgency_level"].lower()
    except json.JSONDecodeError:
        parsed = {"summary_for_staff": raw, "urgency_level": "routine", "missing_info": []}
    return parsed


def medical_handoff_image(pil_image: Image.Image) -> dict:
    """Vision: generate structured medical handoff from a document image."""
    if pil_image is None:
        return {"error": "No image provided"}
    raw = generate_with_image(pil_image, MEDICAL_VISION_PROMPT, max_new_tokens=500)
    try:
        parsed = json.loads(_strip_fences(raw))
        if isinstance(parsed.get("urgency_level"), str):
            parsed["urgency_level"] = parsed["urgency_level"].lower()
    except json.JSONDecodeError:
        parsed = {"summary_for_staff": raw, "urgency_level": "routine", "missing_info": []}
    return parsed


def skills_match(roles: str, certifications: str, languages: str, location: str = "") -> list:
    """Use Gemma 4 to match person's background to the opportunity catalog."""
    if not roles.strip():
        return [{"title": "Enter prior roles above to find matches", "description": "", "location": ""}]

    catalog_text = "\n".join(
        f"- {o['title']} ({o['location']}): {o['description']}"
        for o in OPPORTUNITY_CATALOG
    )
    prompt = (
        f"A displaced person has these skills:\n"
        f"Prior roles: {roles}\n"
        f"Certifications: {certifications or 'None listed'}\n"
        f"Languages: {languages or 'Not specified'}\n\n"
        f"Available humanitarian opportunities:\n{catalog_text}\n\n"
        f"Select the top 3 best matches. Return ONLY valid JSON — a list of objects:\n"
        f'[{{"title": "...", "description": "...", "location": "...", "match_reason": "..."}}]'
    )
    raw = generate_text([{"role": "user", "content": prompt}], max_new_tokens=500)
    try:
        matches = json.loads(_strip_fences(raw))
        if not isinstance(matches, list):
            raise ValueError
    except Exception:
        matches = [{"title": "Could not parse matches", "description": raw[:300], "location": ""}]
    return matches


print("✓ Feature functions ready")

In [ ]:
# ── 8. PDF export ─────────────────────────────────────────────────────────────

def _safe(text) -> str:
    """Encode to Latin-1 safely for Helvetica core font fallback."""
    if text is None:
        return ""
    s = str(text)
    s = s.replace("\u2014", "-").replace("\u2013", "-").replace("\u2019", "'")
    return s.encode("latin-1", errors="replace").decode("latin-1")


def _reshape_arabic(text: str) -> str:
    """Apply Arabic reshaping + bidi so characters connect and render RTL."""
    try:
        import arabic_reshaper
        from bidi.algorithm import get_display
        return get_display(arabic_reshaper.reshape(text))
    except Exception:
        return text


def build_pdf(case: dict, medical: dict | None, skills: list) -> bytes:
    """Build a two-page intake summary PDF (staff copy + client copy)."""
    pdf = FPDF()
    pdf.set_margins(20, 20, 20)
    pdf.add_page()

    # Register Unicode fonts if available
    has_noto = NOTO_SANS is not None
    has_arabic = NOTO_ARABIC is not None
    if has_noto:
        pdf.add_font("Noto", "", NOTO_SANS, uni=True)
    if has_arabic:
        pdf.add_font("NotoAr", "", NOTO_ARABIC, uni=True)

    def section(title):
        pdf.set_font("Helvetica", "B", 12)
        pdf.set_text_color(44, 95, 138)
        pdf.set_x(pdf.l_margin)
        pdf.cell(0, 8, _safe(title), ln=True)
        pdf.set_draw_color(44, 95, 138)
        pdf.line(20, pdf.get_y(), 190, pdf.get_y())
        pdf.ln(2)

    def row(label, value):
        pdf.set_font("Helvetica", "B", 9)
        pdf.set_text_color(80, 80, 80)
        pdf.set_x(pdf.l_margin)
        pdf.cell(55, 6, _safe(label), ln=False)
        pdf.set_font("Helvetica", "", 9)
        pdf.set_text_color(30, 30, 30)
        pdf.cell(0, 6, _safe(value) or "-", ln=True)

    # Header
    pdf.set_font("Helvetica", "B", 10)
    pdf.set_text_color(192, 0, 0)
    pdf.cell(0, 6, "CONFIDENTIAL - STAFF USE ONLY", ln=True)
    pdf.ln(2)
    pdf.set_font("Helvetica", "B", 16)
    pdf.set_text_color(44, 95, 138)
    pdf.cell(0, 10, "RefugeeReach Intake Summary", ln=True)
    pdf.set_font("Helvetica", "", 9)
    pdf.set_text_color(100, 100, 100)
    pdf.cell(0, 5, f"Generated: {datetime.utcnow().strftime('%Y-%m-%d %H:%M UTC')}   Case ID: {case.get('case_id', '-')}", ln=True)
    pdf.ln(4)

    # Registration details
    section("Registration Details")
    row("Name", case.get("person_name"))
    row("Date of Birth", case.get("date_of_birth"))
    row("Gender", case.get("gender"))
    row("Nationality", case.get("nationality"))
    row("Family Size", str(case.get("family_size") or "-"))
    row("Location", case.get("current_location"))
    row("Preferred Language", _LANG_NAMES.get(case.get("language", "en"), case.get("language", "-")))
    flags = ", ".join(case.get("vulnerability_flags") or []) or "None"
    row("Vulnerability Flags", flags)
    pdf.ln(4)

    # Medical handoff
    urgency = (medical or {}).get("urgency_level", "routine") or "routine"
    section(f"Medical Handoff  [{urgency.upper()}]")
    summary = (medical or {}).get("summary_for_staff", "No medical information recorded.")
    pdf.set_font("Helvetica", "", 9)
    pdf.set_text_color(30, 30, 30)
    pdf.set_x(pdf.l_margin)
    pdf.multi_cell(0, 5, _safe(summary) or "-")
    pdf.ln(4)

    # Skills & opportunities
    section("Skills & Opportunities")
    if skills:
        for o in skills[:3]:
            pdf.set_x(pdf.l_margin)
            pdf.set_font("Helvetica", "B", 9)
            pdf.set_text_color(30, 30, 30)
            pdf.multi_cell(0, 5, f"- {_safe(o.get('title', ''))}")
            pdf.set_x(pdf.l_margin)
            pdf.set_font("Helvetica", "", 9)
            pdf.set_text_color(80, 80, 80)
            pdf.multi_cell(0, 5, f"  {_safe(o.get('location', ''))}")
            pdf.ln(1)
    else:
        pdf.set_x(pdf.l_margin)
        pdf.set_font("Helvetica", "", 9)
        pdf.cell(0, 5, "None identified", ln=True)

    # Client copy page
    pdf.add_page()
    pdf.set_font("Helvetica", "B", 14)
    pdf.set_text_color(0, 130, 110)
    pdf.cell(0, 10, "CLIENT COPY - What happens next", ln=True)
    pdf.set_draw_color(0, 130, 110)
    pdf.line(20, pdf.get_y(), 190, pdf.get_y())
    pdf.ln(4)

    # Generate multilingual client message
    lang = case.get("language", "en")
    lang_name = _LANG_NAMES.get(lang, "English")
    name = case.get("person_name") or ""
    greeting = f"for {name}" if name else ""
    client_prompt = [
        {"role": "user", "content": (
            f"You are a kind humanitarian caseworker writing a short message {greeting}.\n"
            f"Write a warm, reassuring 'What happens next' message in {lang_name}. "
            f"Use simple language. Include: (1) registration is complete, "
            f"(2) go to the reception desk for your registration card, "
            f"(3) you can return to this kiosk for help, "
            f"(4) acknowledge their difficult journey and that they are safe here.\n"
            f"Write ONLY the message in {lang_name}. 4 sentences. No headings."
        )}
    ]
    try:
        client_message = generate_text(client_prompt, max_new_tokens=200)
    except Exception:
        client_message = ""

    if client_message:
        if lang in ("ar", "fa") and has_arabic:
            prepared = _reshape_arabic(client_message)
            pdf.set_font("NotoAr", "", 11)
        elif has_noto:
            pdf.set_font("Noto", "", 11)
        else:
            pdf.set_font("Helvetica", "", 10)
            client_message = _safe(client_message)
        pdf.set_text_color(30, 30, 30)
        pdf.set_x(pdf.l_margin)
        pdf.multi_cell(0, 7, client_message)
        pdf.ln(4)

    pdf.set_x(pdf.l_margin)
    pdf.set_font("Helvetica", "I", 7)
    pdf.set_text_color(160, 160, 160)
    pdf.multi_cell(0, 4,
        "Generated by RefugeeReach (Gemma 4, local inference). "
        "All information requires staff verification before official use.")

    return bytes(pdf.output())


print("✓ PDF export ready")

In [ ]:
# ── 9. Gradio UI ──────────────────────────────────────────────────────────────

CSS = """
.rr-header { background: linear-gradient(135deg, #1a3a5c 0%, #2c5f8a 100%); padding: 20px; border-radius: 8px; margin-bottom: 16px; }
.rr-header h1 { color: white; font-size: 1.6rem; margin: 0; }
.rr-header p  { color: #a8c8e8; margin: 4px 0 0 0; font-size: 0.9rem; }
.urgency-urgent   { background: #fee2e2; color: #991b1b; padding: 4px 12px; border-radius: 4px; font-weight: bold; }
.urgency-elevated { background: #fef3c7; color: #92400e; padding: 4px 12px; border-radius: 4px; font-weight: bold; }
.urgency-routine  { background: #d1fae5; color: #065f46; padding: 4px 12px; border-radius: 4px; font-weight: bold; }
"""

URGENCY_COLORS = {"urgent": "#dc2626", "elevated": "#d97706", "routine": "#059669"}

def _format_extracted_fields(extracted: dict) -> str:
    if "error" in extracted:
        return f"Error: {extracted['error']}"
    lines = [f"**Document type:** {extracted.get('document_type', 'unknown').replace('_', ' ').title()}"]
    lines.append(f"**Overall confidence:** {extracted.get('overall_confidence', 0.0):.0%}")
    lines.append(f"\n**Summary:** {extracted.get('summary', '')}")
    lines.append("\n**Extracted fields:**")
    for field, data in (extracted.get("fields") or {}).items():
        if isinstance(data, dict):
            val = data.get("value") or "—"
            conf = data.get("confidence", 0)
            lines.append(f"- **{field.replace('_', ' ').title()}:** {val} *(confidence: {conf:.0%})*")
    return "\n".join(lines)


def _format_medical_result(result: dict) -> str:
    if "error" in result:
        return f"Error: {result['error']}"
    urgency = result.get("urgency_level", "routine")
    color = URGENCY_COLORS.get(urgency, "#059669")
    lines = [f"### Urgency: <span style='color:{color};font-weight:bold'>{urgency.upper()}</span>\n"]
    lines.append(f"**Summary for clinical staff:**\n{result.get('summary_for_staff', '')}")
    missing = result.get("missing_info") or []
    if missing:
        lines.append(f"\n**Missing info to verify:** {', '.join(missing)}")
    for key in ("conditions", "medications", "vaccinations"):
        items = result.get(key) or []
        if items:
            lines.append(f"\n**{key.title()}:** {', '.join(items)}")
    return "\n".join(lines)


def _format_skills_result(matches: list) -> str:
    if not matches:
        return "No matches found."
    lines = []
    for i, m in enumerate(matches, 1):
        lines.append(f"### {i}. {m.get('title', '')}")
        if m.get("location"):
            lines.append(f"📍 *{m['location']}*")
        if m.get("description"):
            lines.append(f"{m['description']}")
        if m.get("match_reason"):
            lines.append(f"\n**Why this matches:** {m['match_reason']}")
        lines.append("")
    return "\n".join(lines)


def _format_case_summary(case: dict) -> str:
    fields = [
        ("Case ID",   case.get("case_id", "—")),
        ("Language",  _LANG_NAMES.get(case.get("language", "en"), "English")),
        ("Name",      case.get("person_name") or "not yet collected"),
        ("DOB",       case.get("date_of_birth") or "—"),
        ("Gender",    case.get("gender") or "—"),
        ("Nationality", case.get("nationality") or "—"),
        ("Family Size", str(case.get("family_size") or "—")),
        ("Location",  case.get("current_location") or "—"),
    ]
    return "\n".join(f"**{k}:** {v}" for k, v in fields)


# ── Gradio Blocks ─────────────────────────────────────────────────────────────
with gr.Blocks(css=CSS, title="RefugeeReach — Gemma 4 Demo") as demo:

    # Header
    gr.HTML("""
    <div class='rr-header'>
      <h1>&#x1F6E1;&#xFE0F; RefugeeReach</h1>
      <p>Multilingual humanitarian intake assistant &mdash; powered by Gemma 4 E4B &middot; Kaggle Gemma 4 Hackathon Demo</p>
    </div>
    """)

    # Shared session state
    case_state    = gr.State(new_case("en"))
    msg_state     = gr.State([])          # full messages list for intake chat
    medical_state = gr.State(None)
    skills_state  = gr.State([])

    with gr.Tabs():

        # ── Tab 1: Welcome ────────────────────────────────────────────────────
        with gr.Tab("🏁 Welcome"):
            gr.Markdown("## Start a new intake session")
            with gr.Row():
                with gr.Column(scale=2):
                    lang_selector = gr.Dropdown(
                        choices=[("English", "en"), ("Arabic / العربية", "ar"),
                                 ("Ukrainian / Українська", "uk"), ("Dari / دری", "fa"),
                                 ("French / Français", "fr")],
                        value="en",
                        label="Preferred language",
                    )
                    start_btn = gr.Button("Start New Case", variant="primary", size="lg")
                with gr.Column(scale=3):
                    welcome_status = gr.Markdown("*Select a language and press Start New Case.*")

            gr.Markdown("---")
            gr.Markdown("""
            ### Quick demo guide (Amira's story)
            1. **Welcome** → Select *Arabic* → Start New Case
            2. **Intake Chat** → `مرحباً، اسمي أميرة حسن، أنا طبيبة أطفال من سوريا`
            3. **Document Vision** → upload any passport image → Extract Fields
            4. **Medical** → symptoms: `persistent cough, night sweats, prior TB exposure`
            5. **Skills** → roles: `pediatric nurse, medical triage` → Find Matches
            6. **Export** → Generate PDF → download
            """)

            def on_start(language):
                case = new_case(language)
                status = (
                    f"✓ Case **{case['case_id']}** created &nbsp;|&nbsp; "
                    f"Language: **{_LANG_NAMES.get(language, language)}**\n\n"
                    f"Proceed to the **Intake Chat** tab to begin registration."
                )
                return case, [], None, [], status

            start_btn.click(
                on_start,
                inputs=[lang_selector],
                outputs=[case_state, msg_state, medical_state, skills_state, welcome_status],
            )

        # ── Tab 2: Intake Chat ────────────────────────────────────────────────
        with gr.Tab("💬 Intake Chat"):
            gr.Markdown("## Multilingual intake conversation")
            gr.Markdown(
                "_Gemma 4 detects the language automatically and responds in kind. "
                "Try Arabic, Ukrainian, Dari, or French — no language switching needed._"
            )
            with gr.Row():
                with gr.Column(scale=3):
                    chatbot = gr.Chatbot(height=420, label="Intake conversation")
                    with gr.Row():
                        chat_input = gr.Textbox(
                            placeholder="Type in any language — or try: مرحباً، اسمي أميرة",
                            label="Your message",
                            lines=2,
                            scale=4,
                        )
                        send_btn = gr.Button("Send", variant="primary", scale=1)

                with gr.Column(scale=2):
                    gr.Markdown("### Case fields collected")
                    case_display = gr.Markdown("*Start a case on the Welcome tab first.*")
                    refresh_btn = gr.Button("Refresh fields", size="sm")

            def on_send(user_msg, chat_history, messages, case):
                if not user_msg.strip():
                    return chat_history, messages, case, "", _format_case_summary(case)
                if not case.get("case_id"):
                    chat_history = chat_history + [[user_msg, "⚠ Please start a case on the Welcome tab first."]]
                    return chat_history, messages, case, "", _format_case_summary(case)
                reply, updated_msgs, updated_case = intake_turn(user_msg, messages, case)
                chat_history = chat_history + [[user_msg, reply]]
                return chat_history, updated_msgs, updated_case, "", _format_case_summary(updated_case)

            send_btn.click(
                on_send,
                inputs=[chat_input, chatbot, msg_state, case_state],
                outputs=[chatbot, msg_state, case_state, chat_input, case_display],
            )
            chat_input.submit(
                on_send,
                inputs=[chat_input, chatbot, msg_state, case_state],
                outputs=[chatbot, msg_state, case_state, chat_input, case_display],
            )
            refresh_btn.click(
                lambda c: _format_case_summary(c),
                inputs=[case_state],
                outputs=[case_display],
            )

        # ── Tab 3: Document Vision ────────────────────────────────────────────
        with gr.Tab("📄 Document Vision"):
            gr.Markdown("## Gemma 4 reads documents in any language")
            with gr.Tabs():

                with gr.Tab("Extract Fields"):
                    gr.Markdown(
                        "Upload a passport, national ID, or any identity document. "
                        "Gemma 4 extracts all fields with confidence scores."
                    )
                    with gr.Row():
                        doc_image_extract = gr.Image(type="pil", label="Document image", height=300)
                        with gr.Column():
                            extract_btn = gr.Button("Extract Fields", variant="primary")
                            extract_output = gr.Markdown("*Upload an image and press Extract Fields.*")

                    def on_extract(img):
                        result = extract_document(img)
                        return _format_extracted_fields(result)

                    extract_btn.click(on_extract, inputs=[doc_image_extract], outputs=[extract_output])

                with gr.Tab("Translate Document"):
                    gr.Markdown(
                        "Upload any document — eviction notice, legal letter, foreign form. "
                        "Gemma 4 explains it in the person's language."
                    )
                    with gr.Row():
                        doc_image_translate = gr.Image(type="pil", label="Document image", height=300)
                        with gr.Column():
                            translate_lang = gr.Dropdown(
                                choices=[("English", "en"), ("Arabic / العربية", "ar"),
                                         ("Ukrainian / Українська", "uk"), ("Dari / دری", "fa"),
                                         ("French / Français", "fr")],
                                value="ar",
                                label="Explain in this language",
                            )
                            translate_btn = gr.Button("Translate & Explain", variant="primary")
                            translate_output = gr.Textbox(
                                label="Explanation",
                                lines=10,
                                placeholder="Upload a document and press Translate & Explain.",
                            )

                    translate_btn.click(
                        translate_document,
                        inputs=[doc_image_translate, translate_lang],
                        outputs=[translate_output],
                    )

        # ── Tab 4: Medical Handoff ────────────────────────────────────────────
        with gr.Tab("🏥 Medical Handoff"):
            gr.Markdown("## Structured triage note for clinical staff")
            gr.Markdown(
                "_Gemma 4 generates a clinical summary with urgency level (routine / elevated / urgent). "
                "Not a diagnosis — a structured handoff for the receiving clinician._"
            )
            with gr.Tabs():

                with gr.Tab("Describe Symptoms"):
                    with gr.Row():
                        with gr.Column():
                            med_symptoms = gr.Textbox(
                                label="Symptoms",
                                lines=3,
                                placeholder="e.g. persistent cough for 3 weeks, night sweats, fatigue",
                            )
                            med_conditions = gr.Textbox(
                                label="Existing conditions (optional)",
                                lines=2,
                                placeholder="e.g. diabetes, asthma, prior TB exposure",
                            )
                            med_medications = gr.Textbox(
                                label="Medications mentioned (optional)",
                                lines=2,
                                placeholder="e.g. metformin 500mg, salbutamol inhaler",
                            )
                            med_text_btn = gr.Button("Generate Handoff Note", variant="primary")
                        with gr.Column():
                            med_text_output = gr.Markdown("*Fill in symptoms and press Generate Handoff Note.*")

                    def on_med_text(symptoms, conditions, medications, case):
                        result = medical_handoff_text(symptoms, conditions, medications)
                        return result, _format_medical_result(result)

                    med_text_btn.click(
                        on_med_text,
                        inputs=[med_symptoms, med_conditions, med_medications, case_state],
                        outputs=[medical_state, med_text_output],
                    )

                with gr.Tab("Upload Medical Document"):
                    gr.Markdown("Upload a prescription, discharge note, or vaccination record.")
                    with gr.Row():
                        med_image = gr.Image(type="pil", label="Medical document", height=300)
                        with gr.Column():
                            med_img_btn = gr.Button("Analyze Document", variant="primary")
                            med_img_output = gr.Markdown("*Upload a document and press Analyze.*")

                    def on_med_image(img):
                        result = medical_handoff_image(img)
                        return result, _format_medical_result(result)

                    med_img_btn.click(
                        on_med_image,
                        inputs=[med_image],
                        outputs=[medical_state, med_img_output],
                    )

        # ── Tab 5: Skills Match ───────────────────────────────────────────────
        with gr.Tab("🔍 Skills Match"):
            gr.Markdown("## Match skills to humanitarian opportunities")
            gr.Markdown(
                "_Gemma 4 reads the person's background and matches them to real roles at the reception center — "
                "so a nurse ends up in the clinic, not the food distribution queue._"
            )
            with gr.Row():
                with gr.Column(scale=2):
                    skills_roles = gr.Textbox(
                        label="Prior roles (comma-separated)",
                        placeholder="e.g. pediatric nurse, medical triage, midwife",
                    )
                    skills_certs = gr.Textbox(
                        label="Certifications (optional)",
                        placeholder="e.g. RN, BLS, Arabic language teacher",
                    )
                    skills_langs = gr.Textbox(
                        label="Languages spoken (optional)",
                        placeholder="e.g. Arabic, English, French",
                    )
                    skills_btn = gr.Button("Find Matches", variant="primary", size="lg")
                with gr.Column(scale=3):
                    skills_output = gr.Markdown("*Enter roles and press Find Matches.*")

            def on_skills(roles, certs, langs):
                matches = skills_match(roles, certs, langs)
                return matches, _format_skills_result(matches)

            skills_btn.click(
                on_skills,
                inputs=[skills_roles, skills_certs, skills_langs],
                outputs=[skills_state, skills_output],
            )

        # ── Tab 6: Export PDF ─────────────────────────────────────────────────
        with gr.Tab("📋 Export PDF"):
            gr.Markdown("## Generate intake summary PDF")
            gr.Markdown(
                "_Generates a two-page PDF: a confidential staff copy (registration + medical + skills) "
                "and a dignified client copy with a 'What happens next' message in the person's language._"
            )
            with gr.Row():
                with gr.Column(scale=2):
                    export_summary = gr.Markdown("*Case summary will appear here.*")
                    export_btn = gr.Button("Generate PDF", variant="primary", size="lg")
                    export_file = gr.File(label="Download PDF", visible=False)
                with gr.Column(scale=3):
                    export_status = gr.Markdown("")

            def on_export(case, medical, skills):
                summary = _format_case_summary(case)
                if not case.get("case_id"):
                    return summary, gr.update(visible=False), "⚠ Start a case on the Welcome tab first."
                try:
                    pdf_bytes = build_pdf(case, medical, skills)
                    path = f"/tmp/refugeereach_{case['case_id']}.pdf"
                    with open(path, "wb") as f:
                        f.write(pdf_bytes)
                    return summary, gr.update(value=path, visible=True), "✓ PDF generated — click to download."
                except Exception as e:
                    return summary, gr.update(visible=False), f"Error: {e}"

            export_btn.click(
                on_export,
                inputs=[case_state, medical_state, skills_state],
                outputs=[export_summary, export_file, export_status],
            )
            # Refresh summary on tab open (via a load trigger)
            export_btn.click(
                lambda c: _format_case_summary(c),
                inputs=[case_state],
                outputs=[export_summary],
            )

print("✓ Gradio UI defined")

In [ ]:
# ── 10. Launch ────────────────────────────────────────────────────────────────
#
# share=True generates a public *.gradio.live URL valid for 72 hours.
# Judges can open this URL in any browser without any local setup.

print("Launching RefugeeReach demo...")
print("The public URL will appear below in ~10 seconds.")
print("Share it with judges — it stays live for 72 hours.\n")

demo.launch(
    share=True,
    show_error=True,
    server_name="0.0.0.0",
)